<a href="https://colab.research.google.com/github/parsonjohnson110-glitch/My-ai-app/blob/main/Quotex_Bot_v4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# ==========================================
# QUOTEX BOT V2
# INSTALL REQUIRED LIBRARIES
# ==========================================

!pip -q install yfinance pandas numpy requests pytz

In [8]:
# ==========================================
# IMPORTS
# ==========================================

import yfinance as yf
import pandas as pd
import numpy as np
import requests
import pytz
import time

from datetime import datetime, timedelta

print("✅ All libraries imported successfully")

✅ All libraries imported successfully


In [9]:
# ==========================================
# TELEGRAM SETTINGS
# ==========================================

BOT_TOKEN = "8736894721:AAEfIHb0KJCePDicNQ1Dcx7DeoF622WJGDw"
CHAT_ID = "6425734827"

def send_signal(message):
    url = f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage"

    payload = {
        "chat_id": CHAT_ID,
        "text": message
    }

    response = requests.post(url, data=payload)

    return response.status_code == 200

print("✅ Telegram module ready")
send_signal("🚀 Quotex Bot V2 is online!")

✅ Telegram module ready


True

In [10]:
# ==========================================
# ASSETS & TIMEZONE
# ==========================================

# Brazil timezone
BRAZIL_TZ = pytz.timezone("America/Sao_Paulo")

# Assets to monitor
ASSETS = {
    "GBP/USD": "GBPUSD=X",
    "AUD/USD": "AUDUSD=X",
    "USD/JPY": "JPY=X",
    "Gold": "GC=F"
}

# Store the last signal sent for each asset
last_signals = {}

print("✅ Assets loaded successfully")

for asset in ASSETS:
    print(f"📈 {asset}")

✅ Assets loaded successfully
📈 GBP/USD
📈 AUD/USD
📈 USD/JPY
📈 Gold


In [11]:
# ==========================================
# INDICATOR ENGINE
# ==========================================

def calculate_indicators(df):

    # Handle MultiIndex columns from newer yfinance versions
    if hasattr(df.columns, "nlevels") and df.columns.nlevels > 1:
        df.columns = df.columns.get_level_values(0)

    # Price data
    close = df["Close"]
    open_price = df["Open"]
    high = df["High"]
    low = df["Low"]

    # SMA
    sma10 = close.rolling(window=10).mean()
    sma20 = close.rolling(window=20).mean()

    # RSI (14)
    delta = close.diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()

    rs = avg_gain / avg_loss.replace(0, 0.000001)

    rsi = 100 - (100 / (1 + rs))

    return {
        "open": open_price,
        "high": high,
        "low": low,
        "close": close,
        "sma10": sma10,
        "sma20": sma20,
        "rsi": rsi
    }

print("✅ Indicator Engine Ready")

✅ Indicator Engine Ready


In [12]:
# ==========================================
# CANDLE PATTERN ENGINE
# ==========================================

def bullish_candle(data):
    """
    Returns True if the latest candle is bullish.
    """
    return data["close"].iloc[-1] > data["open"].iloc[-1]


def bearish_candle(data):
    """
    Returns True if the latest candle is bearish.
    """
    return data["close"].iloc[-1] < data["open"].iloc[-1]


def bullish_rejection(data):
    """
    Bullish rejection (hammer-like candle)
    """
    open_price = data["open"].iloc[-1]
    close_price = data["close"].iloc[-1]
    high = data["high"].iloc[-1]
    low = data["low"].iloc[-1]

    body = abs(close_price - open_price)
    upper_wick = high - max(open_price, close_price)
    lower_wick = min(open_price, close_price) - low

    return (
        close_price > open_price
        and lower_wick > body * 2
        and upper_wick < body
    )


def bearish_rejection(data):
    """
    Bearish rejection (shooting star-like candle)
    """
    open_price = data["open"].iloc[-1]
    close_price = data["close"].iloc[-1]
    high = data["high"].iloc[-1]
    low = data["low"].iloc[-1]

    body = abs(close_price - open_price)
    upper_wick = high - max(open_price, close_price)
    lower_wick = min(open_price, close_price) - low

    return (
        close_price < open_price
        and upper_wick > body * 2
        and lower_wick < body
    )


print("✅ Candle Pattern Engine Ready")

✅ Candle Pattern Engine Ready


In [13]:
# ==========================================
# HIGHER TIMEFRAME TREND FILTER
# ==========================================

def get_higher_trend(ticker):

    try:

        df = yf.download(
            ticker,
            period="5d",
            interval="15m",
            auto_adjust=True,
            progress=False
        )

        # Handle MultiIndex columns
        if hasattr(df.columns, "nlevels") and df.columns.nlevels > 1:
            df.columns = df.columns.get_level_values(0)

        if df.empty or len(df) < 20:
            return "UNKNOWN"

        close = df["Close"]

        sma20 = close.rolling(window=20).mean()

        last_close = close.iloc[-1]
        last_sma20 = sma20.iloc[-1]

        if last_close > last_sma20:
            return "UP"

        elif last_close < last_sma20:
            return "DOWN"

        else:
            return "SIDEWAYS"

    except Exception as e:
        print(f"Trend Filter Error: {e}")
        return "UNKNOWN"


print("✅ Higher Timeframe Filter Ready")

✅ Higher Timeframe Filter Ready


In [14]:
# ==========================================
# SIGNAL BRAIN
# ==========================================

def detect_signal(data, ticker):

    # Make sure we have enough candles
    if len(data["close"]) < 21:
        return None

    close = data["close"]
    sma10 = data["sma10"]
    sma20 = data["sma20"]
    rsi = data["rsi"]

    # Latest values
    current_close = close.iloc[-1]
    current_sma10 = sma10.iloc[-1]
    current_sma20 = sma20.iloc[-1]

    previous_sma10 = sma10.iloc[-2]
    previous_sma20 = sma20.iloc[-2]

    current_rsi = rsi.iloc[-1]

    # Get higher timeframe trend
    trend = get_higher_trend(ticker)

    # -----------------------
    # SMA Crossovers
    # -----------------------

    bullish_cross = (
        previous_sma10 <= previous_sma20
        and current_sma10 > current_sma20
    )

    bearish_cross = (
        previous_sma10 >= previous_sma20
        and current_sma10 < current_sma20
    )

    # -----------------------
    # Candle Patterns
    # -----------------------

    bullish = bullish_candle(data)
    bearish = bearish_rejection(data)

    # -----------------------
    # Price near SMA20
    # -----------------------

    near_sma20 = abs(
        current_close - current_sma20
    ) <= current_close * 0.001

    # ==========================
    # CALL SIGNAL
    # ==========================

    if (
        bullish_cross
        and bullish
        and current_close > current_sma20
        and current_rsi > 50
        and trend == "UP"
    ):
        return "🟢 CALL"

    # ==========================
    # PUT SIGNAL
    # ==========================

    if (
        bearish_cross
        and bearish
        and near_sma20
        and current_rsi < 50
        and trend == "DOWN"
    ):
        return "🔴 PUT"

    return None


print("✅ Signal Brain Ready")

✅ Signal Brain Ready


In [ ]:
# ==========================================
# MONITORING ENGINE
# ==========================================

print("🚀 Quotex Bot V2 Started...")

while True:

    print("\n==============================")
    print("🔍 Scanning Market...")
    print("==============================")

    for asset, ticker in ASSETS.items():

        try:

            # Download latest 5-minute candles
            df = yf.download(
                ticker,
                period="5d",
                interval="5m",
                auto_adjust=True,
                progress=False
            )

            # Handle MultiIndex columns
            if hasattr(df.columns, "nlevels") and df.columns.nlevels > 1:
                df.columns = df.columns.get_level_values(0)

            if df.empty:
                print(f"❌ {asset}: No data")
                continue

            # Calculate indicators
            data = calculate_indicators(df)

            # Detect signal
            signal = detect_signal(data, ticker)

            if signal is None:
                print(f"⏳ {asset}: No signal")
                continue

            # Prevent duplicate alerts
            if last_signals.get(asset) == signal:
                print(f"🔁 {asset}: Duplicate signal skipped")
                continue

            # Time
            now = datetime.now(BRAZIL_TZ)
            expiry = now + timedelta(minutes=5)

            trend = get_higher_trend(ticker)

            # Telegram message
            message = f"""
🚨 QUOTEX SIGNAL 🚨

{signal}

📈 Asset: {asset}

🕒 Signal Time:
{now.strftime('%Y-%m-%d %H:%M:%S')}

⏳ Expiry:
{expiry.strftime('%H:%M:%S')}

📊 Trend:
{trend}

📉 RSI:
{data['rsi'].iloc[-1]:.2f}

📍 SMA10:
{data['sma10'].iloc[-1]:.5f}

📍 SMA20:
{data['sma20'].iloc[-1]:.5f}

🇧🇷 Timezone:
Brazil
"""

            # Send Telegram alert
            if send_signal(message):
                print(f"✅ {asset}: {signal} sent")
                last_signals[asset] = signal
            else:
                print(f"❌ {asset}: Telegram failed")

        except Exception as e:

            print(f"⚠️ {asset}: {e}")

    print("\n⏰ Waiting 5 minutes...\n")

    time.sleep(300)

🚀 Quotex Bot V2 Started...

🔍 Scanning Market...
⏳ GBP/USD: No signal
⏳ AUD/USD: No signal
⏳ USD/JPY: No signal
⏳ Gold: No signal

⏰ Waiting 5 minutes...


🔍 Scanning Market...
⏳ GBP/USD: No signal
⏳ AUD/USD: No signal
⏳ USD/JPY: No signal
⏳ Gold: No signal

⏰ Waiting 5 minutes...


🔍 Scanning Market...
⏳ GBP/USD: No signal
⏳ AUD/USD: No signal
⏳ USD/JPY: No signal
⏳ Gold: No signal

⏰ Waiting 5 minutes...


🔍 Scanning Market...
⏳ GBP/USD: No signal
⏳ AUD/USD: No signal
⏳ USD/JPY: No signal
⏳ Gold: No signal

⏰ Waiting 5 minutes...


🔍 Scanning Market...
⏳ GBP/USD: No signal
⏳ AUD/USD: No signal
⏳ USD/JPY: No signal
⏳ Gold: No signal

⏰ Waiting 5 minutes...


🔍 Scanning Market...
⏳ GBP/USD: No signal
⏳ AUD/USD: No signal
⏳ USD/JPY: No signal
⏳ Gold: No signal

⏰ Waiting 5 minutes...


🔍 Scanning Market...
⏳ GBP/USD: No signal
⏳ AUD/USD: No signal
⏳ USD/JPY: No signal
⏳ Gold: No signal

⏰ Waiting 5 minutes...


🔍 Scanning Market...
⏳ GBP/USD: No signal
⏳ AUD/USD: No signal
⏳ USD/JPY: No